# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



**Finding 1:** Growing content was longer and younger than declining content. Growing pages averaged 3.2K words and 184 days old, compared with 2.3K words and 230 days for declining pages.

**Methodology questions:**

* How was the growing/declining label defined?
* Were the groups comparable in topic, brand, and starting visibility?
* Does this show an association, or does the evidence support a causal claim?

**Finding 2:** 365+ day content that was refreshed within 30 days showed a 3.2× higher health score and 57× more impressions in the dataset.

**Methodology questions:**

* How was a page classified as refreshed?
* Were refreshed pages compared with similar pages that were not refreshed?
* Could the result be due to selection bias rather than the refresh itself?



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [17]:
!git clone https://github.com/Sulamithsingh/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [18]:
import os

path = "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

print(os.path.exists(path))
print(path)

True
/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [19]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load the Week-5 dataset
df = pd.read_csv(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

# Recreate the Week-5 target
df["target"] = (
    (df["days_since_last_update"] >= 90) &
    (df["impressions_90d"] >= 1000) &
    (df["ctr"] <= 0.10) &
    (df["avg_position"] <= 20)
).astype(int)

# Same Week-5 features
features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Keep complete rows
data = df[features + ["target", "client_id"]].dropna()

X = data[features]
y = data["target"]
groups = data["client_id"]

# Client-grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Same Week-5 Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, pred)

print("Week-5 random split accuracy: 1.0")
print("ML-09 client-grouped split accuracy:", honest_accuracy)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Unique clients in training:", groups.iloc[train_idx].nunique())
print("Unique clients in testing:", groups.iloc[test_idx].nunique())

# Check for client overlap
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Clients appearing in both sets:", len(train_clients & test_clients))

Week-5 random split accuracy: 1.0
ML-09 client-grouped split accuracy: 0.9998377413597274
Training rows: 23837
Test rows: 6163
Unique clients in training: 25
Unique clients in testing: 7
Clients appearing in both sets: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [20]:
# ML-09 Leakage Audit

feature_columns = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

target_definition = {
    "days_since_last_update": ">= 90",
    "impressions_90d": ">= 1000",
    "ctr": "<= 0.10",
    "avg_position": "<= 20"
}

print("Target definition:")
print("target = (days_since_last_update >= 90)")
print("         & (impressions_90d >= 1000)")
print("         & (ctr <= 0.10)")
print("         & (avg_position <= 20)")
print()

print("Features used by the model:")
for feature in feature_columns:
    print("-", feature)

print("\nLeakage finding:")
print("The target is directly constructed from all four model features.")
print("Therefore, the model is learning the rule used to create the target.")
print("The 1.0 Week-5 accuracy should not be interpreted as evidence of general predictive performance.")

Target definition:
target = (days_since_last_update >= 90)
         & (impressions_90d >= 1000)
         & (ctr <= 0.10)
         & (avg_position <= 20)

Features used by the model:
- days_since_last_update
- impressions_90d
- ctr
- avg_position

Leakage finding:
The target is directly constructed from all four model features.
Therefore, the model is learning the rule used to create the target.
The 1.0 Week-5 accuracy should not be interpreted as evidence of general predictive performance.


### Leakage audit finding

The Week-5 target is constructed directly from the same four variables used as model features. Therefore, the model has access to the exact information used to define the target.

This explains the perfect Week-5 accuracy and means that the 1.0 score should not be interpreted as evidence that the model can generalize to new data. The main issue is target construction rather than only the train/test split.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [21]:
# ML-09 Error Analysis

results = X_test.copy()
results["actual"] = y_test.values
results["predicted"] = pred

# Find incorrect predictions
errors = results[results["actual"] != results["predicted"]]

print("Total test examples:", len(results))
print("Incorrect predictions:", len(errors))
print("Error rate:", len(errors) / len(results))

print("\nSample errors:")
display(errors.head(10))

Total test examples: 6163
Incorrect predictions: 1
Error rate: 0.00016225864027259452

Sample errors:


,days_since_last_update,impressions_90d,ctr,avg_position,actual,predicted
16859,92,1130,0.09,10.0,1,0


In [22]:
# Show the single incorrect prediction
display(errors)

,days_since_last_update,impressions_90d,ctr,avg_position,actual,predicted
16859,92,1130,0.09,10.0,1,0


### Error analysis

The honest client-grouped test produced one incorrect prediction. The model predicted 0, while the actual target was 1.

The error occurred for a page with 92 days since its last update, 1,130 impressions, a 0.09 CTR, and an average position of 10.0. These values satisfy all four conditions used to define the target.

This is a false negative. The example shows that even though the target is directly constructed from the features, the Random Forest did not reproduce the rule perfectly on this test example. However, this single error is not enough to draw broader conclusions about model weaknesses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.